In [ ]:
try:
    import claveles  # noqa: F401
except ImportError:
    %pip install claveles

**Note for Google Colab Users:**
During installation, Colab may display a dependency conflict error regarding `protobuf` (`ERROR: pip's dependency resolver...`). This occurs due to pre-installed Google packages in the Colab environment. 

As long as you see `Successfully installed ...` at the end of the log, the required packages are properly installed. **Please ignore the error and continue running the rest of the notebook.**

$$
\newcommand{\bra}[1]{{\left\langle{#1}\right|}}
\newcommand{\ket}[1]{{\left|{#1}\right\rangle}}
\newcommand{\Bra}[1]{{\Big\langle{#1}\Big|}}
\newcommand{\Ket}[1]{{\Big|{#1}\Big\rangle}}
\newcommand{\braket}[1]{{\left\langle{#1}\right\rangle}}
\newcommand{\dag}{\dagger}
$$

# Quantum Teleportation with Bosonic Circuits

## Introduction

This demo implements the CV quantum teleportation protocol as a bosonic circuit. Using the Wigner function, we visualize how a Gaussian input state is transferred to another mode through entanglement, a beamsplitter operation, homodyne measurements, and an adaptive feedforward operation.

---

## Background

### The Basic Idea

Quantum teleportation is a quantum information processing protocol that transfers an unknown quantum state from one physical system to another with the aid of quantum entanglement. In a quantum communication setting, where a sender (Alice) wishes to transmit a quantum state to a spatially distant receiver (Bob), teleportation allows for this transfer without physically transporting the particle itself. The two communicators need only to share an entangled resource state beforehand, perform local measurements, communicate the results via a classical channel, and apply local unitary operations conditioned on those outcomes.

The continuous-variable (CV) counterpart of qubit teleportation [\[1\]](#References) was proposed by Braunstein and Kimble (1998) [\[2\]](#References) and experimentally demonstrated by Furusawa et al. in the same year [\[3\]](#References). In the CV version, the quadrature operators of a quantum mode ($\hat{x}$, $\hat{p}$) carry the continuous-variable information, and an arbitrary bosonic state is transferred via homodyne measurement and feedforward.

### CV Teleportation Protocol

The protocol consists of three steps.

**Step 1 — EPR pair preparation**

First, one prepares an entangled resource called the two-mode squeezed vacuum state (or **EPR pair**) between modes 1 and 2 (to be shared between Alice and Bob in quantum communication). The ideal EPR pair $\ket{\mathrm{EPR}}$ is achieved in the limit where the squeezing parameter $r$ of the physical EPR state $\ket{\mathrm{EPR}_r}$ approaches infinity, satisfying:

$$\hat{x}_1 - \hat{x}_2 \to 0, \quad \hat{p}_1 + \hat{p}_2 \to 0.$$

Due to finite energy constraints (which apply similarly to single-mode squeezed states), in practice, we use a finitely correlated approximation of it with a finite squeezing parameter $r$, defined as:

$$\ket{\mathrm{EPR}_r} = \widehat{\mathrm{TS}}(r)\ket{0,0}.$$

Here, $\widehat{\mathrm{TS}}$ denotes the two-mode squeezing gate. This state can also be generated by mixing two single-mode squeezed vacuum states with squeezing parameter $r$ on a balanced beamsplitter as follows:

$$\ket{\mathrm{EPR}_r} \approx \hat{B}_\uparrow (\hat{S}(r, \phi_1) \otimes \hat{S}(r, \phi_2)) \ket{0,0},$$

where $\phi_1 = \pi/2$ ($p$-squeezed), $\phi_2 = 0$ ($x$-squeezed), and $\hat{B}_\uparrow$ denotes the inverse 50:50 beamsplitter operation. (For more details, also see [EPR Tutorial 1](./epr1.ipynb), [2](./epr2.ipynb), and [3](./epr3.ipynb).)

**Step 2 — Homodyne measurement**

Alice prepares an input quantum state $\ket{\psi}$ to be teleported in mode 0 and applies a 50:50 beamsplitter ($\theta = \pi/4$) between modes 0 and 1, then measures each mode's quadrature via homodyne detection:

- Mode 0: measure the $\hat{x}$ quadrature → outcome $x_\text{meas}$
- Mode 1: measure the $\hat{p}$ quadrature → outcome $p_\text{meas}$

Note that once the EPR pair is shared in the first step, this measurement process is strictly local to Alice's side.

**Step 3 — Feedforward displacement**

Once the measurement is executed by Alice, the residual quantum state on Bob's side is conditioned on the measurement outcomes. To cancel out this dependency, Alice communicates the results to Bob via a classical channel, and Bob then applies a corresponding phase-space displacement operation to his state (mode 2):

$$D(\alpha) \quad \text{where} \quad \alpha = \frac{x_\text{meas} - i\, p_\text{meas}}{\sqrt{\hbar}},$$

which shifts $\hat{x}$ by $\sqrt{2}\, x_\text{meas}$ and $\hat{p}$ by $-\sqrt{2}\, p_\text{meas}$ ($D(\alpha)$ shifts $\hat{x}$ by $\sqrt{2\hbar}\,\mathrm{Re}\,\alpha$ and $\hat{p}$ by $\sqrt{2\hbar}\,\mathrm{Im}\,\alpha$).

Accounting for the noise introduced by the finite squeezing resource, this operation yields:

$$\rho_\text{out} \approx \rho_\text{in}$$

effectively transferring the input quantum state onto mode 2 (and the two states become exactly equal in the limit where the squeezing level $r$ approaches infinity). Since Alice's measurement is local, Bob cannot detect the effect of the measurement without receiving the classical information from Alice. This property is a consequence of the **no-signaling theorem** and is one of the fundamental principles of quantum theory; thus, Bob must communicate with Alice to reconstruct the initial input state that she prepared.

---

## Implementation

First, import the required libraries.

In [ ]:
from math import pi, sqrt

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import BeamSplitter, Displacement, MeasureHomodyne
from claveles.circuit.state import QuantumState
from claveles.circuit.visualize import make_figure
from claveles.client import SimulatorClient
from claveles.constant import hbar
from claveles.execute import execute
from numpy import linspace

### Build the circuit

We define a circuit-building function that accepts the squeezing parameter `r` and returns the circuit. This modular design allows us to easily reuse the same circuit architecture for the parameter sweep experiment evaluated below. The input state on mode 0 is the coherent state whose mean vector is `mean`.

The measurement outcomes feed forward into the displacement on mode 2: each homodyne measurement returns a variable that stands for its outcome, and `Displacement` accepts the amplitude $\alpha$ of Step 3 written as an ordinary complex expression over these variables.

In [ ]:
mean = np.array([3.0, -1.5])
input_state = QuantumState.coherent((mean[0] + 1j * mean[1]) / sqrt(2 * hbar))


def build_circuit(r, name="Quantum Teleportation"):
    c = CircuitRepr(name)
    c.Q(0) | input_state
    c.Q(1) | QuantumState.squeezed(r=r, phi=pi / 2)
    c.Q(2) | QuantumState.squeezed(r=r, phi=0)
    c.Q(1, 2) | BeamSplitter(theta=pi / 4, phi=pi)
    c.Q(0, 1) | BeamSplitter(theta=pi / 4, phi=pi)
    measured_x = c.Q(0) | MeasureHomodyne(phi=0)  # x quadrature
    measured_p = c.Q(1) | MeasureHomodyne(phi=pi / 2)  # p quadrature
    alpha = (measured_x - 1j * measured_p) * sqrt(1 / hbar)
    c.Q(2) | Displacement(alpha)
    return c

Let us construct and visualize the target teleportation circuit using a squeezing parameter of $r = 5.0$.

In [ ]:
circuit = build_circuit(r=5.0)

fig = make_figure(circuit)
fig.show()

### Run the simulation

In [ ]:
client = SimulatorClient(backend="cpu", state_save_policy="first_only")
client.n_shots = 1000

result = execute(circuit, client)

---

## Comparing Input and Output via Wigner Functions


The Wigner function serves as a continuous quasi-probability distribution defined over the $(x, p)$ phase space; for an arbitrary Gaussian state, it manifests as a strictly positive, two-dimensional Gaussian distribution. If the teleportation protocol executes with high fidelity, the Wigner function of the input state (mode 0) and that of the final output state (mode 2) should look nearly identical.

In [ ]:
sns.set_theme(style="whitegrid")

xvec = linspace(-10, 10, 401)
pvec = linspace(-10, 10, 401)
final_state = result.client_result.shot_results[0].state

### Input state (mode 0)

The input is chosen as a coherent state centered at phase-space coordinates $(x, p) = (3.0, -1.5)$. Its corresponding Wigner function presents as a positive Gaussian distribution that exhibits the exact same variance as the vacuum state.

In [ ]:
fig = input_state.draw_wigner(xvec, pvec, title="Input state (mode 0): coherent state")
ax = fig.axes[0]
x0, p0 = mean
ax.axvline(x0, color="k", linestyle="--", linewidth=0.8, label=f"peak x={x0}")
ax.axhline(p0, color="k", linestyle=":", linewidth=0.8, label=f"peak p={p0}")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

### Output state (mode 2)

This plot displays the Wigner function of mode 2 extracted post-feedforward correction. The central peak coordinates and the geometric variance match the input state with high precision, visually confirming a faithful quantum state transfer.

In [ ]:
output_mode_2 = final_state.extract_mode(2)
fig = output_mode_2.draw_wigner(xvec, pvec, title="Teleportation output (mode 2)")
ax = fig.axes[0]
x0, p0 = mean
ax.axvline(x0, color="k", linestyle="--", linewidth=0.8, label=f"expected x={x0}")
ax.axhline(p0, color="k", linestyle=":", linewidth=0.8, label=f"expected p={p0}")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

We can quantitatively evaluate the accuracy of the state transfer by computing the Euclidean displacement error between the input and output mean vectors.

In [ ]:
mean_err = np.linalg.norm(mean - output_mode_2.mean.real)

print(f"mean error: {mean_err:.4f}")

Like any Gaussian state, a coherent state is uniquely characterized by two statistical metrics: its mean vector and its covariance matrix. Due to the imperfect quantum correlations of a finitely squeezed EPR resource, the covariance matrix $V_\text{out}$ of the unconditional output state (averaged over all homodyne measurement outcomes) is related to the input state's covariance matrix $V_\text{in}$ by an additive noise term. Meanwhile, the averaged mean vector remains invariant:
$$V_\text{out} = V_\text{in} + \hbar e^{-2r}I, \quad \mu_\text{out} = \mu_\text{in}$$
where $I$ denotes the $2\times 2$ identity matrix. In any single run of the protocol, however, the conditional mean vector undergoes a random shift dictated by the specific measurement outcome; statistically averaging these random shifts is exactly what gives rise to the additive noise seen in the unconditional covariance matrix.

---

## Squeezing Strength and Transfer Quality

As discussed in the last section, the overall state transfer fidelity of CV quantum teleportation depends heavily on the squeezing parameter $r$ configuring the initial ancilla resource states.

### Squeezing and EPR Correlations

The continuous quadrature variances characterizing a squeezed vacuum state (configured here for $x$-squeezing) scale as follows:

$$\Delta x^2 = e^{-2r} \cdot \frac{\hbar}{2}, \quad \Delta p^2 = e^{+2r} \cdot \frac{\hbar}{2}$$

Increasing the parameter $r$ exponentially suppresses quantum fluctuations along the squeezed quadrature axis. Since a physical EPR state is generated by mixing two single-mode squeezed states at a 50:50 beam splitter, the strength of the quadrature correlations of the resulting EPR state can be directly related to the variance of the initial squeezed states:

$$\Delta(x_1 - x_2)^2 \propto e^{-2r}, \quad \Delta(p_1 + p_2)^2 \propto e^{-2r}$$

where $(x_i, p_i)$ denotes the quadratures of mode $i$ ($i = 1, 2$) of the EPR state. These correlations approach ideal EPR entanglement as $r \to \infty$, reducing the relative fluctuations to zero in the asymptotic limit.

### Effect on Transfer Quality

A larger value of $r$ tightens the underlying EPR correlations, which directly suppresses the stochastic displacement errors introduced during the real-time feedforward stage of the teleportation protocol. In the limit $r \to \infty$, the quantum correlations become perfectly sharp, enabling the output state to exactly replicate the input state. For energy-bounded implementations at a finite value of $r$, the residual EPR correlation noise induces a random displacement error in the conditional output mean vector on a run-by-run basis. The magnitude (or variance) of this stochastic error decreases monotonically as the squeezing strength $r$ is enhanced.

The following code executes a parameter sweep across multiple values of $r$, reporting the resulting variance of the displacement error in the final mean vector.

In [ ]:
for r in [0.5, 1.0, 2.0, 5.0]:
    circuit_r = build_circuit(r=r, name=f"Teleportation r={r}")
    result_r = execute(circuit_r, client)
    final_mode_r = result_r.client_result.shot_results[0].state.extract_mode(2)

    err = np.linalg.norm(mean - final_mode_r.mean.real)
    print(f"r={r:.1f}  mean vector error: {err:.4f}")

---

## Summary

This tutorial demonstrated the following:

1. **Continuous-variable teleportation protocol**: An arbitrary input bosonic state can be faithfully transferred across distinct modes in three consecutive stages: Entangled EPR pair generation, joint homodyne measurement, and active feedforward displacement correction.
2. **Phase-space verification via the Wigner function**: The Wigner function characterizing the output mode post-feedforward matches the initial input state profile with high precision, providing clear visual confirmation of a successful quantum state transfer.
3. **Squeezing dependence of transfer fidelity**: Increasing the squeezing parameter $r$ tightens the underlying inter-mode EPR correlations and suppresses the feedforward displacement error, systematically approaching ideal, deterministic quantum teleportation in the asymptotic limit $r \to \infty$.

---

## References

- \[1\] C. H. Bennett et al., Teleporting an unknown quantum state via dual classical and Einstein-Podolsky-Rosen channels, [*Phys. Rev. Lett.* **70**, 1895 (1993)](https://doi.org/10.1103/PhysRevLett.70.1895).
- \[2\] S. L. Braunstein and H. J. Kimble, Teleportation of Continuous Quantum Variables, [*Phys. Rev. Lett.* **80**, 869 (1998)](https://doi.org/10.1103/PhysRevLett.80.869).
- \[3\] A. Furusawa et al., Unconditional Quantum Teleportation, [*Science* **282**,706-709(1998)](https://www.science.org/doi/10.1126/science.282.5389.706).